In [1]:
import os
import json
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("API"))

In [12]:
import json

def load_user_messages(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    username = data.get("username")
    
    # Try to get 'timeline' (new format), fallback to 'messages' (old format)
    raw_messages = data.get("timeline") or data.get("messages") or []
    
    # Clean the messages: keep only 'type' and 'text' to match your LLM's expectations
    cleaned_messages = [
        {
            "type": m.get("type"),
            "text": m.get("text")
        } 
        for m in raw_messages
    ]
    
    return username, cleaned_messages

In [3]:
def build_analysis_prompt(username, messages):

    message_block = "\n".join([f"{i+1}. {msg}" for i, msg in enumerate(messages)])

    prompt = f"""
You are analyzing Reddit posts/comments from ONE user account.

Username: {username}

Messages written by this account:

{message_block}

Analyze the messages collectively to detect patterns, intent, and likelihood of automation.

Tasks:

1. Identify whether the messages promote a repeated theme, narrative, ideology, or running joke.

2. Classify the primary content intent:

- spam (advertising, scams, promotion, link farming)
- propaganda (political or ideological narrative pushing)
- normal_discussion (general conversation across varied topics)
- running_bit (repeated joke, meme persona, or intentionally recurring theme)

3. Determine whether the account appears to be:
- human_user
- likely_bot

Important:
Spam or propaganda can be posted by both humans and bots.  
When estimating automation likelihood, focus on signals such as:
- repetitive or templated messages
- identical structure across posts
- lack of conversational engagement
- copy-paste style content
- systematic or mechanical tone.

4. Estimate the probability that the account is automated.

5. If likely_bot, classify the most probable bot type:

- spam_bot
- propaganda_bot
- karma_farming_bot
- engagement_bot
- unknown_bot

Return ONLY JSON in the following format:

{{
"content_classification": "spam | propaganda | normal_discussion | running_bit",
"account_type": "human_user | likely_bot",
"pattern_detected": true/false,
"pattern_description": "...",
"bot_probability": 0.0,
"bot_type": "spam_bot | propaganda_bot | karma_farming_bot | engagement_bot | unknown_bot",
"key_indicators": [
"indicator1",
"indicator2",
"indicator3"
]
}}
"""
    return prompt

In [13]:
def analyze_user(cleaned_messages, username):

    prompt = build_analysis_prompt(username, cleaned_messages)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [14]:
username, cleaned_messages = load_user_messages("reddit_user_Plus-Affect-6365_scraped.json")

In [15]:
result = analyze_user(cleaned_messages, username)

print(result)

```json
{
  "content_classification": "normal_discussion",
  "account_type": "human_user",
  "pattern_detected": true,
  "pattern_description": "The user frequently engages in discussions related to specific video games, primarily Genshin Impact (e.g., weapon choices, character builds, team compositions, wish mechanics) and Clash of Clans strategies. Another prominent theme in their posts and comments is academic and career development, focusing on AI/Machine Learning (e.g., resume reviews, higher education applications, technical concepts like autoencoders/CNNs, Leetcode). They also show interest in general tech (Android phones) and scam awareness.",
  "bot_probability": 0.01,
  "bot_type": "unknown_bot",
  "key_indicators": [
    "Engages in detailed and context-specific discussions across multiple complex domains (e.g., Genshin Impact character rotations, ML algorithms, Clash of Clans attack strategies).",
    "Exhibits natural language, including self-correction ('LOFL' to 'LOFI'),